In [9]:
import duckdb

print(duckdb.sql("""
    SELECT DISTINCT
        recipient_uei,
        recipient_name,
        recipient_address_line_1,
        recipient_city_name,
        recipient_state_code,
        LEFT(recipient_zip_4_code, 5) AS zip,
        federal_action_obligation,
        lat,
        lon
    FROM 'master_geocoded.parquet'
    WHERE match = 'Manual'
    LIMIT 20
"""))

#WHERE recipient_uei = 'JJL8JCR5RT31'

┌───────────────┬──────────────────────────────────────────────────────────────────┬───────────────────────────┬─────────────────────┬──────────────────────┬─────────┬───────────────────────────┬─────────────────┬───────────────────┐
│ recipient_uei │                          recipient_name                          │ recipient_address_line_1  │ recipient_city_name │ recipient_state_code │   zip   │ federal_action_obligation │       lat       │        lon        │
│    varchar    │                             varchar                              │          varchar          │       varchar       │       varchar        │ varchar │          varchar          │     double      │      double       │
├───────────────┼──────────────────────────────────────────────────────────────────┼───────────────────────────┼─────────────────────┼──────────────────────┼─────────┼───────────────────────────┼─────────────────┼───────────────────┤
│ L4ZMKLM4MPL3  │ MOOG INC                                      

In [5]:
import duckdb

con = duckdb.connect()

df = con.execute("""
    SELECT * FROM 'master_cleaned.parquet' LIMIT 0
""").fetchdf()

print(df.columns.tolist())

['contract_transaction_unique_key', 'contract_award_unique_key', 'award_id_piid', 'modification_number', 'federal_action_obligation', 'total_dollars_obligated', 'total_outlayed_amount_for_overall_award', 'current_total_value_of_award', 'potential_total_value_of_award', 'action_date', 'action_date_fiscal_year', 'period_of_performance_start_date', 'period_of_performance_current_end_date', 'awarding_agency_code', 'awarding_agency_name', 'awarding_sub_agency_code', 'awarding_sub_agency_name', 'awarding_office_code', 'awarding_office_name', 'funding_agency_code', 'funding_agency_name', 'funding_sub_agency_code', 'funding_sub_agency_name', 'recipient_uei', 'recipient_duns', 'recipient_name', 'recipient_name_raw', 'recipient_parent_uei', 'recipient_parent_duns', 'recipient_parent_name', 'recipient_country_code', 'recipient_country_name', 'recipient_address_line_1', 'recipient_city_name', 'recipient_county_name', 'recipient_state_code', 'recipient_state_name', 'recipient_zip_4_code', 'prime_aw

In [6]:
import duckdb

con = duckdb.connect()

con.execute("""
COPY (
    SELECT
        m.*,
        g.lat,
        g.lon,
        g.match,
        u.address_id
    FROM 'master_cleaned.parquet' m

    LEFT JOIN read_csv_auto('uei_name_lookup.csv') u
        ON m.clean_uei = u.uei
        AND m.clean_name = u.name
        AND m.clean_address = u.address
        AND m.clean_city = u.city
        AND m.clean_state = u.state
        AND m.clean_zip = u.zip

    LEFT JOIN read_csv_auto('geocoded_addresses_from_script.csv') g
        ON u.address_id = g.id

)
TO 'master_geocoded.parquet'
(FORMAT PARQUET);
""")

con.close()

In [2]:
import duckdb

con = duckdb.connect()

con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE prime_award_transaction_recipient_cd_current IS NULL) AS missing_cd,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE prime_award_transaction_recipient_cd_current IS NULL) / COUNT(*),
        2
    ) AS missing_cd_percent
FROM 'master_geocoded.parquet'
""").fetchdf()

,total_rows,missing_cd,missing_cd_percent
0,55764141,31715,0.06


In [8]:
con.execute("""
SELECT
    COUNT(*) AS unmatched_geocodes,
    
    COUNT(*) FILTER (WHERE prime_award_transaction_recipient_cd_current IS NOT NULL) AS unmatched_with_cd,
    
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE prime_award_transaction_recipient_cd_current IS NOT NULL) / COUNT(*),
        2
    ) AS percent_with_cd
    
FROM 'master_geocoded.parquet'
WHERE (match != 'Match' AND match != 'Manual')
   OR match IS NULL
""").fetchdf()

,unmatched_geocodes,unmatched_with_cd,percent_with_cd
0,4632868,4629508,99.93


In [7]:
con.execute("""
SELECT COUNT(*) AS distinct_addresses
FROM ( 
  SELECT DISTINCT
    clean_uei,
    clean_name,
    clean_address,
    clean_city,
    clean_state,
    clean_zip,
    match,
    lat,
    lon
  FROM 'master_geocoded.parquet'
  WHERE ((match != 'Match' AND match != 'Manual') OR match IS NULL)
    AND prime_award_transaction_recipient_cd_current IS NULL
)
""").fetchdf()

,distinct_addresses
0,192


In [10]:
con.execute("""
SELECT
    clean_uei,
    clean_name,
    clean_address,
    clean_city,
    clean_state,
    clean_zip,
    COUNT(*) AS transactions,
    SUM(TRY_CAST(federal_action_obligation AS DECIMAL(18,2))) AS total_obligation
FROM 'master_geocoded.parquet'
WHERE ((match != 'Match' AND match != 'Manual') OR match IS NULL)
  AND prime_award_transaction_recipient_cd_current IS NULL
GROUP BY
    clean_uei,
    clean_name,
    clean_address,
    clean_city,
    clean_state,
    clean_zip
ORDER BY total_obligation DESC
""").fetchdf()

,clean_uei,clean_name,clean_address,clean_city,clean_state,clean_zip,transactions,total_obligation
0,QN1BCFY7JDJ5,RTX CORPORATION,400 MAIN ST,EAST HARTFORD,FL,06108,517,7.313157e+08
1,C76NUJZDXL75,WSP USA ENVIRONMENT & INFRASTRUCTURE INC.,ONE PLYMOUTH MEETING,PLYMOUTH MEETI,PA,19462,221,1.491642e+08
2,DJKNQ7AWDQL8,"TRAX INTERNATIONAL, LLC",WHITE SANDS MR,WHITE SANDS MI,NM,88002,80,1.167585e+08
3,EJGKYVW1NLL5,"POLARIS ALPHA, LLC",5450 TECH CENTER DR,COLORADO SPRIN,CO,80919,117,7.590016e+07
4,ENNHFPHSDEX4,PERATON INC.,8955A DRENNAN RD,COLORADO SPRIN,CO,80925,31,5.257173e+07
...,...,...,...,...,...,...,...,...
187,M3J1B3ML1CE5,"CRANE ELECTRONICS, INC.",84 HILL AVE,FORT WALTON BE,FL,32548,6,-4.297360e+05
188,TLJZWFZZQMF6,"M2 TECHNOLOGIES, INC.",498 ELLIOTT RD CENTERVI,WEST HYANNISPO,MA,02672,8,-5.273063e+05
189,V6Y5MMVCJLL5,"KELLOGG BROWN & ROOT SERVICES, INC.",2316 BRANDENBURG STATION R,FORT KNOX,MI,40121,1,-7.851400e+05
190,JE47BHSEWSP4,ADVANCED TECHNICAL PRODUCTS INC,ONE GARVIES PT RD,GLEN COVE,GA,11542,22,-1.066736e+06


In [23]:
con.execute("""
SELECT DISTINCT
  clean_uei,
  clean_name,
  clean_address,
  clean_city,
  clean_state,
  clean_zip
FROM 'master_geocoded.parquet'
WHERE (clean_zip IS NULL OR TRIM(clean_zip) = '')
  AND prime_award_transaction_recipient_cd_current IS NOT NULL
  AND lat IS NULL
  AND lon IS NULL
""").fetchdf()

,clean_uei,clean_name,clean_address,clean_city,clean_state,clean_zip
0,CNMRBP7GV796,TELEGUAM HOLDINGS LLC,624 N MARINE CORPS DR,TAMUNING,GU,None
1,GPXJYERQRVK4,NAVIG8 CHEMICALS POOL INC.,TRUST COMPANY COMPLEX AJELTAK,MAJURO,MH,None
2,GAXHMJHP2YS1,"GUAM CABLEVISION, LLC",600 HARMON LOOP RD,DEDEDO 96929,GU,None
3,S7QSS3NZXE26,DALLAS ENTERPRISES,AMELCO BUILDING RT 15 - PAGAT,MANGILAO,GU,None
4,DN6DLL9MNNX5,KWIKSPACE GUAM INC,CHALAN SAN ANTONIO LOT 2145,"AGANA, 96932",GU,None
5,F8F6D1RJ4ST3,TRINITY EVANGELICAL DIVINITY S,2065 HALF DAY RD,DEERFIELD,IL,None
6,MFELCMEAUVS5,"LESCANO, JOSEPH L",1270 NORTH MARNE DR,TAMUNING,GU,None
7,RQ7EQKRLK5Q9,ST MATTHEWS LIMITED LIABILITY COMPANY,SE 101 2025 ARMY DR,DEDEDO,GU,None
8,DN6DLL9MNNX5,KWIKSPACE GUAM INC,256 MARINE CORPS DR ROUTE 1,PITI,GU,None
9,FGS7N9Q9MMW5,INTERNATIONAL BRIDGE CORPORATION,171 MARINE CORP DR,YIGO 96929,GU,None


In [6]:
con.execute("""
SELECT
    COUNT(DISTINCT bad.clean_uei || '|' || bad.clean_address) AS unmatched_unique_addresses,
    COUNT(DISTINCT CASE WHEN good.lat IS NOT NULL THEN bad.clean_uei || '|' || bad.clean_address END) AS addresses_with_other_geocode
FROM 'master_geocoded.parquet' bad

LEFT JOIN 'master_geocoded.parquet' good
    ON bad.clean_uei = good.clean_uei
    AND bad.clean_address = good.clean_address
    AND good.lat IS NOT NULL

WHERE (bad.match != 'Match' OR bad.match IS NULL)
""").fetchdf()

,unmatched_unique_addresses,addresses_with_other_geocode
0,29664,250


In [11]:
con.execute("""
WITH bad AS (
    SELECT DISTINCT
        clean_uei,
        clean_city
    FROM 'master_geocoded.parquet'
    WHERE ((match != 'Match' AND match != 'Manual') OR match IS NULL)
),

good AS (
    SELECT DISTINCT
        clean_uei,
        clean_city
    FROM 'master_geocoded.parquet'
    WHERE lat IS NOT NULL
)

SELECT
    COUNT(*) AS unmatched_unique_uei_city,
    COUNT(g.clean_uei) AS with_other_geocode
FROM bad b
LEFT JOIN good g
    ON b.clean_uei = g.clean_uei
    AND b.clean_city = g.clean_city
""").fetchdf()

,unmatched_unique_uei_city,with_other_geocode
0,26478,4593


In [5]:
con.execute("""
COPY (
    SELECT
        bad.* EXCLUDE (lat, lon, match),

        COALESCE(bad.lat, good.lat) AS lat,
        COALESCE(bad.lon, good.lon) AS lon,

        CASE
            WHEN bad.lat IS NULL AND good.lat IS NOT NULL THEN 'Manual'
            ELSE bad.match
        END AS match

    FROM 'master_geocoded.parquet' bad

    LEFT JOIN (
        SELECT DISTINCT
            clean_uei,
            clean_address,
            lat,
            lon
        FROM 'master_geocoded.parquet'
        WHERE lat IS NOT NULL
    ) good
        ON bad.clean_uei = good.clean_uei
        AND bad.clean_address = good.clean_address

)
TO 'master_geocoded.parquet'
(FORMAT PARQUET);
""")

In [6]:
import duckdb

con = duckdb.connect()

con.execute("""
COPY (
    SELECT
        bad.* EXCLUDE (lat, lon, match),

        COALESCE(bad.lat, good.lat) AS lat,
        COALESCE(bad.lon, good.lon) AS lon,

        CASE
            WHEN bad.lat IS NULL AND good.lat IS NOT NULL THEN 'Manual'
            ELSE bad.match
        END AS match

    FROM 'master_geocoded.parquet' bad

    LEFT JOIN (
        SELECT
            clean_uei,
            clean_city,
            ANY_VALUE(lat) AS lat,
            ANY_VALUE(lon) AS lon
        FROM 'master_geocoded.parquet'
        WHERE lat IS NOT NULL
        GROUP BY
            clean_uei,
            clean_city
        HAVING COUNT(DISTINCT lat || ',' || lon) = 1
    ) good
        ON bad.clean_uei = good.clean_uei
        AND bad.clean_city = good.clean_city

)
TO 'master_geocoded.parquet'
(FORMAT PARQUET);
""")